In [1]:
# Assign Shepard classes based on sand/silt/clay percentages

import numpy as np
import pandas as pd
from pathlib import Path
import warnings
from matplotlib.path import Path as MplPath

SQRT3 = np.sqrt(3.0)

# ============================================================
# Ternary transform
#   sand = lower left
#   silt = lower right
#   clay = top
# ============================================================
def ternary_to_xy(sand, silt, clay):
    sand = np.asarray(sand, dtype=float)
    silt = np.asarray(silt, dtype=float)
    clay = np.asarray(clay, dtype=float)

    total = sand + silt + clay
    if not np.allclose(total, 100.0):
        raise ValueError("Each point must sum to 100.")

    x = (silt + 0.5 * clay) / 100.0
    y = (SQRT3 / 2.0) * clay / 100.0
    return x, y


def ternary_polygon_to_xy(points):
    pts = np.asarray(points, dtype=float)
    x, y = ternary_to_xy(pts[:, 0], pts[:, 1], pts[:, 2])
    return np.column_stack([x, y])


# ============================================================
# Shepard class polygons based on the boundaries we defined
# ============================================================
def build_shepard_polygons():
    polys = {
        "sand": [
            (100, 0, 0),
            (75, 25, 0),
            (75, 0, 25),
        ],
        "silt": [
            (0, 100, 0),
            (0, 75, 25),
            (25, 75, 0),
        ],
        "clay": [
            (0, 0, 100),
            (25, 0, 75),
            (0, 25, 75),
        ],
        "silty sand": [
            (75, 25, 0),
            (50, 50, 0),
            (40, 40, 20),
            (60, 20, 20),
            (75, 12.5, 12.5),
        ],
        "sandy silt": [
            (25, 75, 0),
            (12.5, 75, 12.5),
            (20, 60, 20),
            (40, 40, 20),
            (50, 50, 0),
        ],
        "clayey sand": [
            (75, 0, 25),
            (75, 12.5, 12.5),
            (60, 20, 20),
            (40, 20, 40),
            (50, 0, 50),
        ],
        "clayey silt": [
            (0, 75, 25),
            (50, 0, 50),  # placeholder to be replaced below
        ],
        "sandy clay": [
            (25, 0, 75),
            (12.5, 12.5, 75),
            (20, 20, 60),
            (40, 20, 40),
            (50, 0, 50),
        ],
        "silty clay": [
            (0, 25, 75),
            (0, 50, 50),
            (20, 40, 40),
            (20, 20, 60),
            (12.5, 12.5, 75),
        ],
        "sand-silt-clay": [
            (60, 20, 20),
            (20, 60, 20),
            (20, 20, 60),
        ],
    }

    # Correct the mirrored right-side polygon explicitly
    polys["clayey silt"] = [
        (0, 75, 25),
        (0, 50, 50),
        (20, 40, 40),
        (20, 60, 20),
        (12.5, 75, 12.5),
    ]

    return polys


# ============================================================
# Build matplotlib Paths for point-in-polygon classification
# ============================================================
def build_polygon_paths():
    polygons = build_shepard_polygons()
    paths = {}

    for name, poly in polygons.items():
        xy = ternary_polygon_to_xy(poly)
        paths[name] = MplPath(xy)

    return paths


# ============================================================
# Classify one sample
# ============================================================
def classify_shepard_sample(sand, silt, clay, polygon_paths, atol=1e-6):
    total = sand + silt + clay
    if not np.isclose(total, 100.0, atol=atol):
        return "unclassified"

    x, y = ternary_to_xy(sand, silt, clay)
    pt = (float(x), float(y))

    # Order matters only for exact boundary cases.
    # This order keeps the central and corner classes clear.
    class_order = [
        "sand-silt-clay",
        "sand",
        "silt",
        "clay",
        "silty sand",
        "sandy silt",
        "clayey sand",
        "clayey silt",
        "sandy clay",
        "silty clay",
    ]

    for cls in class_order:
        if polygon_paths[cls].contains_point(pt, radius=1e-12):
            return cls

    return "unclassified"


# ============================================================
# Read, warn, classify, and write output
# ============================================================
infile = Path("shepard_test.csv")
outfile = Path("shepard_test_classes.csv")

if not infile.exists():
    raise FileNotFoundError(f"Could not find input file: {infile}")

df = pd.read_csv(infile)

required = ["id", "sand", "silt", "clay"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Input CSV is missing required columns: {missing}")

# Warn if any rows do not sum to 100%
row_sums = df[["sand", "silt", "clay"]].sum(axis=1)
bad = ~np.isclose(row_sums.values, 100.0, atol=1e-6)

if bad.any():
    msg = [f"Rows in {infile.name} do not sum to 100%:"]
    for i in np.where(bad)[0]:
        msg.append(
            f"  row {i}: id={df.loc[i, 'id']}, "
            f"sand={df.loc[i, 'sand']}, "
            f"silt={df.loc[i, 'silt']}, "
            f"clay={df.loc[i, 'clay']}, "
            f"sum={row_sums.iloc[i]}"
        )
    warnings.warn("\n".join(msg))

# Build class polygons and classify
polygon_paths = build_polygon_paths()

df["shepard_class"] = [
    classify_shepard_sample(sand, silt, clay, polygon_paths)
    for sand, silt, clay in zip(df["sand"], df["silt"], df["clay"])
]

# Write output
df.to_csv(outfile, index=False)

print(f"Wrote classified file: {outfile}")
print(df[["id", "sand", "silt", "clay", "shepard_class"]].head())

Wrote classified file: shepard_test_classes.csv
   id  sand  silt  clay shepard_class
0   1    72     3    25   clayey sand
1   2    65    25    10    silty sand
2   3    23     4    73    sandy clay
3   4     3    23    74    silty clay
4   5    18    72    10    sandy silt
